In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [ ]:
# dataloader_binary.py 먼저 만들기
# model_binary_resnet.py로 모델 구조
# train_binary.py 작성해서 학습
# 성능 확인 후 → 필요 시 증강 적용 또는 ResNet34/Dropout 확장
# 사용 모델 RetNet18 모델

In [ ]:
# RetNet18 모델 구현
"""
torchvision.models.resnet18(pretrained=True) 로 사전학습 모델 사용
- ImageNet에 미리 학습된 가중치(weigth) 사용 : 1000개 클래스, 120만장 이미지 분류 데이터셋
  (전이학습)
마지막 fc 층을 클래스 수 2개 (food, not_food)에 맞게 교체
깔끔하게 함수로 정의해서 학습 스크립트에서 불러와 쓰기 좋게 구성
"""

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import os

# 앞서 설정한 함수 불러오기
from dataloader_binary import get_binary_dataloaders
from model_binary_resnet import get_resnet18_binary_model

# 하이퍼 파라미터 설정
batch_size = 32
num_epoch = 15
learning_rate = 1e-4
svae_path = "best_binary_model.pth"

# GPU 장치 설정
device = torch.devicece("cuda" if torch.cuda.is_available() else "cpu")
model = get_resnet18_binary_model().to(device)

# 데이터 불러오기
train_loader, val_loader, _ = get_binary_dataloaders(
    data_dir = "데이터 원본",   # ★★★★★ 경로 수정 필요 (어떤경로인지?)
    batch_size = batch_size,
    num_workers = 2 # 이건 4로 해도 되는지?
)

# 모델 불러오기 
model = get_resnet18_binary_model(pretrained=True).to(device)

# 손실 함수 & 옵티마이저 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.paramters()m, lr=learning_rate)

# 학습 루프
best_val_acc = 0.0
for epoch in range(num_epoch):
    print(f"\n [Epoch {epoch+1}/{num_epoch}]")
    
    # Trian Phase
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    
    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    train_acc = correct/total
    avg_train_loss = train_loss/total
    print(f"Train Loss:{avg_train_loss:.4f} | Train Acc : {train_acc:.4f}")
    
    # Validataion Phase
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    
    for images, labels in tqdm(val_loss, desc="Validataion"):
        imgaes,labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        val_loss+= loss.item() * images.size(0)
        pteds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    val_acc = correct/total
    avg_val_loss = val_los / total
    print(f"Val Loss : {avg_val_loss:.4f} | Val Acc : {val_acc:.4f}")
    
    # 최고 성능 모델 저장
    if val_acc > best_val_acc :
        best_val_acc = val_acc
        torch.save(model.state_dict(),save_path)
        print(f"Best model saved to : {save_path}")